# 15 · Connecting vision to a language model: projectors, token splicing, pixel shuffle, resamplers, M-RoPE

**Needed for:** LLaVA (vision track, step V5) and Qwen2-VL (step V6). Janus (step V8) reuses the splicing.

A VLM is: vision encoder (a ViT, notebook 13) → **connector** → language model. The language model sees image
patches as extra "tokens" inside its input sequence. Everything tricky happens at that boundary:

1. **Projector:** map vision features to the LLM's hidden size.
2. **Splicing:** put the image embeddings where the placeholder tokens are, and exclude them from the loss.
3. **Token reduction:** 576–2000+ tokens per image is expensive, so merge neighbours (pixel shuffle) or compress
   to a fixed number of learned queries (resampler / Q-Former).
4. **Positions:** 1-D positions throw away the 2-D layout. Qwen2-VL's M-RoPE gives image tokens (time, row,
   column) positions.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. LLaVA projector and splicing image tokens into text

LLaVA-1.5 uses a 2-layer MLP projector. The tokenizer expands one `<image>` marker into $N$ placeholder ids (one
per image token), so the sequence length already accounts for the image. Then:
- replace the embeddings at placeholder positions with the projected image features, in order;
- set their labels to -100 (ignored by cross-entropy), along with the prompt. Only the answer is trained.

In [ ]:
IMAGE_TOKEN = 999


class Projector(nn.Module):
    def __init__(self, d_vision, d_llm):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_vision, d_llm), nn.GELU(), nn.Linear(d_llm, d_llm))

    def forward(self, x):
        return self.net(x)


def merge_image_tokens(text_embeds, input_ids, image_feats):
    """text_embeds: (B, T, D) embeddings of input_ids (placeholder rows are garbage).
    image_feats: (n_images_total, N, D) projected features, in reading order across the batch.
    Returns (B, T, D) with each placeholder row replaced by the next image feature row."""
    mask = input_ids == IMAGE_TOKEN
    assert mask.sum() == image_feats.shape[0] * image_feats.shape[1], "placeholder count != image tokens"
    out = text_embeds.clone()
    out[mask] = image_feats.reshape(-1, image_feats.shape[-1]).to(out.dtype)
    return out


def build_labels(input_ids, answer_start):
    """Train only on answer tokens: everything before answer_start[b] (prompt + image) is -100."""
    labels = input_ids.clone()
    positions = torch.arange(input_ids.shape[1])[None]
    labels[positions < answer_start[:, None]] = -100
    labels[input_ids == IMAGE_TOKEN] = -100
    return labels


def test_merge(fn):
    torch.manual_seed(0)
    N, D = 4, 8
    ids = torch.tensor([[1, 2] + [IMAGE_TOKEN] * N + [5, 6],
                        [IMAGE_TOKEN] * N + [3, 4, 7, 8]])
    emb = torch.randn(2, 8, D)
    imgs = torch.randn(2, N, D)
    out = fn(emb, ids, imgs)
    torch.testing.assert_close(out[0, 2:6], imgs[0])
    torch.testing.assert_close(out[1, 0:4], imgs[1])
    torch.testing.assert_close(out[0, [0, 1, 6, 7]], emb[0, [0, 1, 6, 7]])     # text untouched


test_merge(merge_image_tokens)
ids = torch.tensor([[1, 2] + [IMAGE_TOKEN] * 4 + [5, 6, 7]])
print("labels:", build_labels(ids, answer_start=torch.tensor([7])).tolist())

## 2. Pixel shuffle: 4× fewer image tokens

InternVL and others merge each 2×2 group of neighbouring patch features into one token by stacking their channels:
$(h, w, D) \to (h/2, w/2, 4D)$, then the projector maps $4D \to D_{llm}$. No information is dropped; the LLM
just sees fewer, wider tokens.

In [ ]:
def pixel_shuffle(x, h, w, r=2):
    """x: (B, h*w, D) row-major patch features -> (B, (h/r)*(w/r), r*r*D)."""
    B, _, D = x.shape
    x = x.reshape(B, h // r, r, w // r, r, D).permute(0, 1, 3, 2, 4, 5)   # (B, h/r, w/r, r, r, D)
    return x.reshape(B, (h // r) * (w // r), r * r * D)


def test_pixel_shuffle(fn):
    h, w, D = 4, 6, 3
    x = torch.arange(h * w * D).float().reshape(1, h * w, D)
    out = fn(x, h, w)
    assert out.shape == (1, 6, 12)
    grid = x.reshape(h, w, D)
    expected_first = torch.cat([grid[0, 0], grid[0, 1], grid[1, 0], grid[1, 1]])   # the top-left 2x2 block
    torch.testing.assert_close(out[0, 0], expected_first)


test_pixel_shuffle(pixel_shuffle)
print("24x24 patches:", 576, "tokens -> after pixel shuffle:", pixel_shuffle(torch.randn(1, 576, 8), 24, 24).shape[1])

## 3. Resampler: a fixed number of learned queries

Flamingo's Perceiver Resampler and BLIP-2's Q-Former use $M$ learned query vectors that cross-attend to all
image features. Output is always $M$ tokens (32–64), whatever the resolution. Cheaper for the LLM, but it can lose
fine detail (text in images, small objects), which is why most recent VLMs went back to MLP projectors plus
pixel shuffle.

In [ ]:
class Resampler(nn.Module):
    def __init__(self, d, n_queries=32, heads=4):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.norm_kv = nn.LayerNorm(d)

    def forward(self, feats):
        q = self.queries.expand(feats.shape[0], -1, -1)
        kv = self.norm_kv(feats)
        x = q + self.attn(q, kv, kv, need_weights=False)[0]
        return x + self.ff(x)


for n_patches in (256, 1024):
    print(f"{n_patches} patches -> {Resampler(64)(torch.randn(2, n_patches, 64)).shape[1]} tokens")

## 4. M-RoPE (Qwen2-VL): 3-D positions

Each token gets three position ids: (temporal, height, width).
- Text tokens: all three equal to the ordinary 1-D position, so for pure text it reduces to normal RoPE.
- Image tokens (one frame): temporal = start, height = start + row, width = start + column.
- The text after an image continues from `max(previous positions) + 1`, so a 24×24 image uses only 24 positions
  of the sequence's position budget instead of 576.

The rotary frequencies of each head are split into three sections; each section rotates by one of the three ids.
Qwen2-VL uses sections of (16, 24, 24) frequency pairs for head_dim 128.

In [ ]:
def mrope_position_ids(segments):
    """segments: list of ("text", n_tokens) or ("image", (grid_h, grid_w)). Returns (3, T) long tensor."""
    ids, start = [], 0
    for kind, size in segments:
        if kind == "text":
            p = torch.arange(start, start + size)
            ids.append(torch.stack([p, p, p]))
            start += size
        else:
            gh, gw = size
            rows = torch.arange(gh)[:, None].expand(gh, gw).flatten()
            cols = torch.arange(gw)[None, :].expand(gh, gw).flatten()
            ids.append(torch.stack([torch.full_like(rows, start), start + rows, start + cols]))
            start = start + max(gh, gw)
    return torch.cat(ids, dim=1)


def mrope_angles(pos3, head_dim, sections, base=10000.0):
    """pos3: (3, T). sections: frequency pairs per component, sum = head_dim // 2. Returns (T, head_dim // 2)."""
    assert sum(sections) == head_dim // 2
    inv_freq = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)
    component = torch.repeat_interleave(torch.arange(3), torch.tensor(sections))   # which id drives each pair
    return pos3[component].T.float() * inv_freq                                     # (T, hd/2)


def test_mrope_ids(fn):
    ids = fn([("text", 3), ("image", (2, 3)), ("text", 2)])
    assert ids.shape == (3, 11)
    assert ids[:, :3].tolist() == [[0, 1, 2]] * 3
    assert ids[0, 3:9].tolist() == [3] * 6                          # one frame: same temporal id
    assert ids[1, 3:9].tolist() == [3, 3, 3, 4, 4, 4]               # rows
    assert ids[2, 3:9].tolist() == [3, 4, 5, 3, 4, 5]               # columns
    assert ids[:, 9:].tolist() == [[6, 7]] * 3                      # text resumes at max + 1 = 3 + 3


test_mrope_ids(mrope_position_ids)
text_only = mrope_position_ids([("text", 10)])
plain = torch.outer(torch.arange(10).float(), 1.0 / 10000 ** (torch.arange(0, 128, 2).float() / 128))
torch.testing.assert_close(mrope_angles(text_only, 128, (16, 24, 24)), plain)
print("M-RoPE on pure text == ordinary RoPE")
print(mrope_position_ids([("text", 2), ("image", (2, 2)), ("text", 2)]))

## Things to be able to explain

- The usual LLaVA training recipe: stage 1 trains only the projector on caption data (vision and LLM frozen);
  stage 2 unfreezes the LLM for instruction data. Why this order?
- Dynamic resolution: Qwen2-VL feeds native-resolution images (any grid size) through the ViT with 2-D RoPE,
  while LLaVA-NeXT tiles the image into fixed-size crops plus a thumbnail. Trade-offs?
- Where do most VLM bugs come from? (Placeholder count vs feature count mismatches, wrong label masking,
  image features in the wrong dtype.)

## Drill

In [ ]:
def my_merge_image_tokens(text_embeds, input_ids, image_feats):
    raise NotImplementedError


def my_pixel_shuffle(x, h, w, r=2):
    raise NotImplementedError


def my_mrope_position_ids(segments):
    raise NotImplementedError


drill(test_merge, my_merge_image_tokens)
drill(test_pixel_shuffle, my_pixel_shuffle)
drill(test_mrope_ids, my_mrope_position_ids)